# FTS ML Hyperparameter Optimization & Model Registry Workspace

Welcome to the interactive hyperparameter optimization and model registry workspace! This notebook shows you how to:
1. **Load database connections and settings** for FTS.
2. **Dynamically explore available model types** and configurations in the registry.
3. **Programmatically load composable specs** (`HParamStudySpec`) built with component imports (`specs/components/...`).
4. **Run Optuna hyperparameter search** using the core `hparam_search` engine.
5. **Visualize search results** using native Optuna Plotly graphs.
6. **Inspect candidate models** in a pandas DataFrame, and **promote** the best model to production.

### 1. Import Dependencies and Initialize DB Connections

In [1]:
import os
import yaml
from nets.spec import HParamStudySpec
import pandas as pd
import optuna
from datetime import datetime, timezone

from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.repository import MarketDataRepository, ModelRepository
from trading_bot.core.schemas import BarData

from nets.training.hparam_search import run_hparam_search, TRAINER_REGISTRY
from nets.models import NNTrainingConfig

# Initialize tables using application configuration settings
init_db(extra_models=["trading_bot.core.models"])
print("Database engine initialized. Available schemas prepared.")

Database engine initialized. Available schemas prepared.


### 2. Dynamic Model Registry Exploration

To adhere to the Open/Closed Principle (OCP), we retrieve the available model architectures dynamically from the codebase's central registry map. Adding any new model trainer or configuration in the core code automatically registers and exposes it here.

In [2]:
available_models = list(TRAINER_REGISTRY.keys())
print("Available model types in TRAINER_REGISTRY:", available_models)

Available model types in TRAINER_REGISTRY: ['lstm', 'rnn', 'cnn', 'linear_regression', 'xgboost']


### 3. Load & Inspect Composable Search Specifications

We load the composable YAML search configuration file `specs/train/BTCUSDT/lstm_hparam_search.yaml` using `HParamStudySpec.from_yaml()`. Built on the `BaseComposableSpec` engine and strongly-typed Pydantic sub-models (`OptunaStudySpec`, `MarketSpec`, `DateRangeSpec`, `FeatureSetSpec`), it automatically resolves root-anchored component imports (`specs/components/...`) and exposes typed properties directly.

In [3]:
CHOSEN_MODEL_TYPE = "xgboost"
CHOSEN_MARKET = "BTCUSDT"
CHOSEN_FEATURE_SET = "ohlcv"

config_path = f"./specs/train/{CHOSEN_MARKET}/{CHOSEN_MODEL_TYPE}_hparam_{CHOSEN_FEATURE_SET}.yaml"
spec = HParamStudySpec.from_yaml(config_path)

print(f"Loaded HParamStudySpec from: {config_path}")
print(f"  Study Spec   : {spec.study}")
print(f"  Market Spec  : {spec.market}")
print(f"  Dates Spec   : {spec.dates}")
print(f"  Features Spec: {spec.features}")
print(f"  Search Space : {list(spec.search_space.keys())}")

Loaded HParamStudySpec from: ./specs/train/BTCUSDT/xgboost_hparam_ohlcv.yaml
  Study Spec   : study_name='presentation_xgboost_all_features' direction='minimize' n_trials=10 model_type='xgboost'
  Market Spec  : instrument_id='BTC/USDT' interval='30m'
  Dates Spec   : start_date=datetime.datetime(2025, 11, 4, 19, 30, tzinfo=TzInfo(0)) end_date=datetime.datetime(2026, 5, 30, 19, 30, tzinfo=TzInfo(0))
  Features Spec: lookback_period=20 feature_cols=['open', 'high', 'low', 'close', 'volume'] feature_pipeline={'class_path': 'trading_bot.core.transforms.FeaturePipeline', 'params': {'transforms': [{'class_path': 'trading_bot.core.transforms.LogReturnTransform', 'params': {'col_idx': 3}}, {'class_path': 'trading_bot.core.transforms.RatioTransform', 'params': {'num_idx': 1, 'den_idx': 3}}, {'class_path': 'trading_bot.core.transforms.RatioTransform', 'params': {'num_idx': 2, 'den_idx': 3}}, {'class_path': 'trading_bot.core.transforms.RatioTransform', 'params': {'num_idx': 0, 'den_idx': 3}}, {'

### 4. Configuration Validation

We use the strongly-typed sub-model schemas (LSP/ISP validation) to dynamically validate that all hyperparameter keys specified under `spec.search_space` match expected parameter fields in the model configuration class and core trainer configurations.

In [4]:
# Validate loaded configurations dynamically
model_type = spec.study.model_type
if model_type not in TRAINER_REGISTRY:
    raise ValueError(f"Model type '{model_type}' is invalid. Supported: {available_models}")

trainer_cls, config_cls = TRAINER_REGISTRY[model_type]
print(f"Validating search space params against {config_cls.__name__} & NNTrainingConfig...")

model_fields = set(config_cls.model_fields.keys())
nn_fields = set(NNTrainingConfig.model_fields.keys())
all_valid_fields = model_fields.union(nn_fields)

search_space = spec.search_space
for param in search_space:
    if param not in all_valid_fields:
        print(f"⚠️  WARNING: Parameter '{param}' is not defined in the core model configuration classes.")
    else:
        print(f"  - Parameter '{param}' validated successfully.")

Validating search space params against XGBoostConfig & NNTrainingConfig...
  - Parameter 'max_depth' validated successfully.
  - Parameter 'n_estimators' validated successfully.
  - Parameter 'learning_rate' validated successfully.
  - Parameter 'subsample' validated successfully.
  - Parameter 'colsample_bytree' validated successfully.
⚠️  WARNING: Parameter 'min_child_weight' is not defined in the core model configuration classes.
  - Parameter 'gamma' validated successfully.
  - Parameter 'reg_alpha' validated successfully.
  - Parameter 'reg_lambda' validated successfully.


### 4.1. Visualize Raw and Preprocessed Training Data

We load the historical bar data from the database using `spec.market` and `spec.dates` sub-models, apply the feature transformation pipeline, and visualize both the raw price series and the stationary log return input features.

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trading_bot.core.dataset import DatasetBuilder
from trading_bot.core.transforms import BaseTransform
import numpy as np

# 1. Fetch raw bar data using core repository and spec sub-models
instrument_id = spec.market.instrument_id
interval = spec.market.interval
start_date = spec.dates.start_date
end_date = spec.dates.end_date

with SessionLocal() as db:
    market_repo = MarketDataRepository(db)
    raw_bars = market_repo.get_bars(
        instrument_id,
        interval=interval,
        start_date=start_date,
        end_date=end_date,
    )

# 2. Convert to DataFrame
df_viz = pd.DataFrame([{
    "timestamp": b.timestamp,
    "open": b.open,
    "high": b.high,
    "low": b.low,
    "close": b.close,
    "volume": b.volume
} for b in raw_bars])

# 3. Dynamically extract feature columns and execute FeaturePipeline from spec
feature_cols = spec.features.feature_cols
matrix_raw = df_viz[feature_cols].values

if spec.features.feature_pipeline:
    pipeline = BaseTransform.from_dict(spec.features.feature_pipeline)
    features = pipeline.fit_transform(matrix_raw)
    # If transformation reduces length (e.g. log returns diff), pad with NaNs to align with timestamps
    diff_rows = len(df_viz) - len(features)
    if diff_rows > 0:
        pad = np.full((diff_rows, features.shape[1]), np.nan)
        features_aligned = np.vstack([pad, features])
    else:
        features_aligned = features
else:
    pipeline = None
    features_aligned = matrix_raw

# 4. Dynamically plot original price and transformed features using Plotly
num_features = features_aligned.shape[1]
subplot_titles = [f"Raw Close Price ({instrument_id})"]
for i in range(num_features):
    if pipeline and hasattr(pipeline, "transforms") and i < len(pipeline.transforms):
        t_name = pipeline.transforms[i].__class__.__name__
        subplot_titles.append(f"Feature {i + 1}: {t_name}")
    else:
        col_label = feature_cols[i] if i < len(feature_cols) else f"Feature {i + 1}"
        subplot_titles.append(f"Feature {i + 1}: {col_label}")

fig_viz = make_subplots(
    rows=num_features + 1, cols=1, shared_xaxes=True,
    subplot_titles=subplot_titles
)

# Raw price series trace
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["close"], name="Close Price", line=dict(color="#2196F3")), row=1, col=1)

# Feature traces
colors = ["#FF9800", "#4CAF50", "#F44336", "#9C27B0", "#795548", "#00BCD4", "#E91E63"]
for i in range(num_features):
    color = colors[i % len(colors)]
    label = subplot_titles[i + 1]
    fig_viz.add_trace(
        go.Scatter(x=df_viz["timestamp"], y=features_aligned[:, i], name=label, line=dict(color=color)),
        row=i + 2, col=1
    )

fig_viz.update_layout(height=250 * (num_features + 1), title_text="Feature Pipeline Overview (Loaded from Spec)", showlegend=False)
fig_viz.show()

### 5. Execute Hyperparameter Optimization

We invoke `run_hparam_search` directly using our spec object (or file path). Optuna will evaluate different parameter sets, train candidates, log metrics to TensorBoard, and register model metadata to our registry.

In [6]:
print(f"Starting Optuna study '{spec.study.study_name}' ({spec.study.n_trials} trials)...\n")
run_hparam_search(spec)
print("\nHyperparameter optimization study complete!")

Starting Optuna study 'presentation_xgboost_all_features' (10 trials)...



[I 2026-09-20 23:15:13,491] Using an existing study with `study_name='presentation_xgboost_all_features'` instead of creating a new one.
[I 2026-09-20 23:15:13,672] Trial 13 finished with value: 1.0060390195576474e-05 and parameters: {'max_depth': 8, 'n_estimators': 58, 'learning_rate': 0.18835300947523778, 'subsample': 0.7729143539292478, 'colsample_bytree': 0.711067226005216, 'min_child_weight': 7, 'gamma': 0.4966253551547682, 'reg_alpha': 1.3889784487261818e-05, 'reg_lambda': 0.0014242918235297928}. Best is trial 2 with value: 5.8655145949160215e-06.
[I 2026-09-20 23:15:13,982] Trial 14 finished with value: 1.0069763447972946e-05 and parameters: {'max_depth': 2, 'n_estimators': 108, 'learning_rate': 0.10981534115996827, 'subsample': 0.8332012729678088, 'colsample_bytree': 0.8203966516614822, 'min_child_weight': 8, 'gamma': 0.2622802364672309, 'reg_alpha': 0.884764462288041, 'reg_lambda': 0.0009307290288960652}. Best is trial 2 with value: 5.8655145949160215e-06.
[I 2026-09-20 23:15:

Best trial: 2 with loss 5.8655145949160215e-06

Hyperparameter optimization study complete!


### 6. Plot Optuna Optimization Visualizations

We load the Optuna study programmatically from SQLite using `spec.study.study_name` and render native interactive plots using Optuna's native Plotly backend.

In [7]:
optuna_storage = settings.DATABASE_URL.replace("sqlite+pysqlite://", "sqlite://")
try:
    study = optuna.load_study(study_name=spec.study.study_name, storage=optuna_storage)
    print(f"Loaded study '{study.study_name}' containing {len(study.trials)} trials.")
    print(f"Best trial: {study.best_trial.number} | Best Value: {study.best_value}")

    # Render native Plotly plots
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()

    if len(study.trials) > 1:
        fig2 = optuna.visualization.plot_param_importances(study)
        fig2.show()
        
        fig3 = optuna.visualization.plot_slice(study)
        fig3.show()
except Exception as e:
    print("Could not load or plot Optuna study visualizations:", e)

Loaded study 'presentation_xgboost_all_features' containing 23 trials.
Best trial: 2 | Best Value: 5.8655145949160215e-06


/tmp/ipykernel_156725/625118269.py:12: UserWarning: Some regimes for parameter `max_depth` have less than 2 trials. The importance of the parameter may be inaccurate.
  fig2 = optuna.visualization.plot_param_importances(study)
/tmp/ipykernel_156725/625118269.py:12: UserWarning: Some regimes for parameter `n_estimators` have less than 2 trials. The importance of the parameter may be inaccurate.
  fig2 = optuna.visualization.plot_param_importances(study)
/tmp/ipykernel_156725/625118269.py:12: UserWarning: Some regimes for parameter `reg_alpha` have less than 2 trials. The importance of the parameter may be inaccurate.
  fig2 = optuna.visualization.plot_param_importances(study)
/tmp/ipykernel_156725/625118269.py:12: UserWarning: Some regimes for parameter `learning_rate` have less than 2 trials. The importance of the parameter may be inaccurate.
  fig2 = optuna.visualization.plot_param_importances(study)
/tmp/ipykernel_156725/625118269.py:12: UserWarning: Some regimes for parameter `subsa

### 7. Inspect Model Registry Candidates

We use pandas to fetch all registered candidates from our database model registry table (`model_registry`) filtering by `spec.study.model_type`, `spec.market.instrument_id`, and `spec.market.interval`. This provides a tabular dashboard of all run trials, hyperparameters, and validation metrics.

In [8]:
from nets.training.hparam_search import get_scored_models

with SessionLocal() as db:
    df_showcase = get_scored_models(
        db,
        model_type=spec.study.model_type,
        instrument_id=spec.market.instrument_id,
        interval=spec.market.interval
    )

# Display showcase dataframe
display_cols = [
    "model_id", "model_type", "instrument_id", "interval", 
    "val_loss", "ic", "directional_accuracy", "composite_score", "status", "created_at"
]
df_showcase[display_cols].head(15)

,model_id,model_type,instrument_id,interval,val_loss,ic,directional_accuracy,composite_score,status,created_at
5,52d30070281e,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.600000,candidate,2026-09-21 02:15:14
0,0991117f8b47,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.571072,candidate,2026-09-21 02:15:13
6,ba46caa1ff16,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.504925,candidate,2026-09-21 02:15:14
3,e286c87e45e5,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.486295,candidate,2026-09-21 02:15:14
1,0b287517ddee,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.463986,candidate,2026-09-21 02:15:13
4,9dda35368799,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.440025,candidate,2026-09-21 02:15:14
2,319d2f0dd600,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.421488,candidate,2026-09-21 02:15:14
7,f29e43e023fa,xgboost,BTC/USDT,30m,0.00001,0.0,0.559322,0.400000,candidate,2026-09-21 02:15:14


### 8. Programmatic Model Promotion

To promote a model to production status, copy the `model_id` from the DataFrame above and paste it below. The repository will demote any active production model sharing the same signature and promote the selected candidate.

In [9]:
# --- ENTER THE MODEL ID TO PROMOTE ---
model_id_to_promote = ""  # e.g., "model_lstm_btc_usd_..."

if model_id_to_promote:
    with SessionLocal() as db:
        repo = ModelRepository(db)
        repo.promote_to_production(model_id_to_promote)
        db.commit()
    print(f"Model '{model_id_to_promote}' successfully promoted to PRODUCTION status.")
    
    # Display status verification
    with SessionLocal() as db:
        df_verify = pd.read_sql(f"SELECT model_id, status, onnx_path FROM model_registry WHERE model_id='{model_id_to_promote}'", db.bind)
    print("\nUpdated database status:")
    print(df_verify)
else:
    print("Please copy/paste a valid 'model_id' into 'model_id_to_promote' to promote it.")

Please copy/paste a valid 'model_id' into 'model_id_to_promote' to promote it.
